In [1]:
from datasets import load_dataset


In [2]:
# ARCD: Arabic Reading Comprehension Dataset
print("Loading ARCD dataset...")
arcd = load_dataset("hsseinmz/arcd", trust_remote_code=True)

print(f"Train: {len(arcd['train'])} | Validation: {len(arcd['validation'])}")
print("\nSample:")
s = arcd['train'][0]
print(f"Question : {s['question']}")
print(f"Context  : {s['context'][:200]}...")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'hsseinmz/arcd' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'hsseinmz/arcd' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Loading ARCD dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/174k [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/192k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/693 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/702 [00:00<?, ? examples/s]

Train: 693 | Validation: 702

Sample:
Question : - من هو جمال أحمد حمزة خاشقجي؟
Context  : جمال أحمد حمزة خاشقجي (13 أكتوبر 1958، المدينة المنورة - 2 أكتوبر 2018)، صحفي وإعلامي سعودي، رأس عدّة مناصب لعدد من الصحف في السعودية، وتقلّد منصب مستشار، كما أنّه مدير عام قناة العرب الإخبارية سابقًا...


In [3]:
from sentence_transformers import InputExample, losses
from torch.utils.data import DataLoader

SEARCH_MODEL_NAME = "aubmindlab/bert-base-arabertv02"

# Build (question, passage) positive pairs from ARCD
def build_search_pairs(dataset_split, max_samples=5000):
    pairs = []
    for i, item in enumerate(dataset_split):
        if i >= max_samples:
            break
        question = item["question"]
        context  = item["context"]
        if question and context:
            pairs.append(InputExample(texts=[question, context], label=1.0))
    return pairs

train_pairs = build_search_pairs(arcd["train"], max_samples=8000)
print(f"✅ Training pairs: {len(train_pairs)}")
print(f"   Example Q: {train_pairs[0].texts[0]}")
print(f"   Example P: {train_pairs[0].texts[1][:100]}...")

✅ Training pairs: 693
   Example Q: - من هو جمال أحمد حمزة خاشقجي؟
   Example P: جمال أحمد حمزة خاشقجي (13 أكتوبر 1958، المدينة المنورة - 2 أكتوبر 2018)، صحفي وإعلامي سعودي، رأس عدّ...


/tmp/ipykernel_613/4049374022.py:1: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import InputExample, losses


In [4]:
from sentence_transformers import SentenceTransformer, losses
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Load pre-trained Arabic BERT as bi-encoder
search_model = SentenceTransformer(SEARCH_MODEL_NAME, device=DEVICE)

train_dataloader = DataLoader(train_pairs, shuffle=True, batch_size=16)

# Multiple Negatives Ranking Loss — in-batch negatives (very effective for retrieval)
mnr_loss = losses.MultipleNegativesRankingLoss(model=search_model)

SEARCH_EPOCHS   = 3
WARMUP_STEPS    = int(len(train_dataloader) * SEARCH_EPOCHS * 0.1)

print(f"🚀 Fine-tuning Arabic bi-encoder for {SEARCH_EPOCHS} epochs...")
search_model.fit(
    train_objectives=[(train_dataloader, mnr_loss)],
    epochs=SEARCH_EPOCHS,
    warmup_steps=WARMUP_STEPS,
    output_path="./arabert-semantic-search",
    show_progress_bar=True
)
print("✅ Semantic search model saved to ./arabert-semantic-search")

config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/381 [00:00<?, ?B/s]

...

🚀 Fine-tuning Arabic bi-encoder for 3 epochs...


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Semantic search model saved to ./arabert-semantic-search


In [6]:
from sentence_transformers import util

def evaluate_retrieval(model, eval_data, k=5, n_samples=500):
    """
    Evaluate retrieval quality using Precision@K and Recall@K.
    For each question, the correct passage should be in top-K results.
    """
    questions = []
    passages  = []
    for i, item in enumerate(eval_data):
        if i >= n_samples:
            break
        questions.append(item["question"])
        passages.append(item["context"])

    print(f"Encoding {len(questions)} questions and passages...")
    q_embeddings = model.encode(questions, batch_size=32, show_progress_bar=True, normalize_embeddings=True)
    p_embeddings = model.encode(passages,  batch_size=32, show_progress_bar=True, normalize_embeddings=True)

    # Cosine similarity matrix
    cos_scores = util.cos_sim(q_embeddings, p_embeddings)  # (n_q, n_p)

    hits_at_k = 0
    for i in range(len(questions)):
        top_k_indices = cos_scores[i].topk(k).indices.tolist()
        if i in top_k_indices:   # correct passage has same index as question
            hits_at_k += 1

    precision_at_k = hits_at_k / len(questions)
    recall_at_k    = hits_at_k / len(questions)   # same for single-answer tasks

    print(f"\n📊 Retrieval Evaluation (K={k}, N={len(questions)} samples)")
    print(f"   Precision@{k} : {precision_at_k:.4f} ({hits_at_k}/{len(questions)})")
    print(f"   Recall@{k}    : {recall_at_k:.4f}")
    return precision_at_k, recall_at_k


# Load the fine-tuned model for evaluation
finetuned_search_model = SentenceTransformer("./arabert-semantic-search", device=DEVICE)
precision, recall = evaluate_retrieval(finetuned_search_model, arcd["validation"], k=5)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Encoding 500 questions and passages...


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]


📊 Retrieval Evaluation (K=5, N=500 samples)
   Precision@5 : 0.7420 (371/500)
   Recall@5    : 0.7420
